# FairWarn-SHS — Fairness Diagnostics

Runs the selected all-edge GraphSAGE model across five seeds and measures subgroup gaps for gender, school type, residence, and programme.

In [ ]:

!pip -q install torch-geometric pandas numpy scikit-learn matplotlib


In [ ]:

from google.colab import files
uploaded = files.upload()

# Upload both:
# FairWarn_SHS_Node_Features.csv
# FairWarn_SHS_Edge_List.csv


In [ ]:

import random
from copy import deepcopy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, balanced_accuracy_score,
    accuracy_score, confusion_matrix
)
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

NODE_FILE = "FairWarn_SHS_Node_Features.csv"
EDGE_FILE = "FairWarn_SHS_Edge_List.csv"
SEEDS = [42, 123, 456, 789, 1010]
ATTRIBUTES = {
    "Gender": "Q1_Gender",
    "School_Type": "Q4_SchoolType",
    "Residence": "Q6_Residence",
    "Programme": "Q5_Programme",
}
MIN_GROUP_TEST_N = 5

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

nodes = pd.read_csv(NODE_FILE)
edges = pd.read_csv(EDGE_FILE)

labelled_mask = (
    nodes["Label_Available"].eq(1) &
    nodes["TARGET_AtRisk"].notna()
).to_numpy()

node_map = {node_id: i for i, node_id in enumerate(nodes["Node_ID"])}

pairs = []
for _, row in edges.iterrows():
    s_id, t_id = row["Source_Node_ID"], row["Target_Node_ID"]
    if s_id in node_map and t_id in node_map:
        s, t = node_map[s_id], node_map[t_id]
        pairs.extend([(s, t), (t, s)])

edge_index = torch.tensor(pairs, dtype=torch.long).t().contiguous()

excluded = {
    "Node_ID", "Roster_Code", "School_Code", "Class_Code",
    "Label_Available", "TARGET_AtRisk"
}
features = [c for c in nodes.columns if c not in excluded]
X_raw = nodes[features].copy()

num_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_raw.columns if c not in num_cols]

for c in num_cols:
    X_raw[c] = X_raw[c].fillna(X_raw[c].median())

for c in cat_cols:
    mode = X_raw[c].mode(dropna=True)
    X_raw[c] = X_raw[c].fillna(mode.iloc[0] if not mode.empty else "Missing")

processor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
])

X = processor.fit_transform(X_raw).astype(np.float32)
y = nodes["TARGET_AtRisk"].fillna(-1).astype(int).to_numpy()

data = Data(
    x=torch.tensor(X, dtype=torch.float32),
    edge_index=edge_index,
    y=torch.tensor(y, dtype=torch.long)
)

print("Nodes:", data.num_nodes)
print("Labelled nodes:", int(labelled_mask.sum()))
print("Undirected edges:", edge_index.shape[1] // 2)
print("Encoded features:", data.num_node_features)

group_count_rows = []
for name, col in ATTRIBUTES.items():
    subset = nodes.loc[labelled_mask, [col, "TARGET_AtRisk"]].copy()
    subset[col] = subset[col].fillna("Missing").astype(str)
    for group_name, g in subset.groupby(col):
        group_count_rows.append({
            "Attribute": name,
            "Group": group_name,
            "Labelled_N": len(g),
            "AtRisk_N": int(g["TARGET_AtRisk"].eq(1).sum()),
            "NotAtRisk_N": int(g["TARGET_AtRisk"].eq(0).sum()),
            "AtRisk_Rate": float(g["TARGET_AtRisk"].mean())
        })
group_counts_df = pd.DataFrame(group_count_rows)

def make_masks(seed):
    idx = np.where(labelled_mask)[0]
    y_lab = y[idx]
    train_val, test = train_test_split(
        idx, test_size=0.20, stratify=y_lab, random_state=seed
    )
    train_val_y = y[train_val]
    train, val = train_test_split(
        train_val, test_size=0.1875,
        stratify=train_val_y, random_state=seed
    )
    masks = []
    for part in [train, val, test]:
        mask = torch.zeros(len(y), dtype=torch.bool)
        mask[part] = True
        masks.append(mask)
    return masks

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, 64, aggr="mean")
        self.conv2 = SAGEConv(64, 32, aggr="mean")
        self.classifier = torch.nn.Linear(32, 2)
        self.dropout = 0.35

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.classifier(x)

def train_seed(seed):
    set_seed(seed)
    train_mask, val_mask, test_mask = make_masks(seed)
    graph = data.clone()
    graph.train_mask, graph.val_mask, graph.test_mask = train_mask, val_mask, test_mask
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    graph = graph.to(device)
    model = GraphSAGE(graph.num_node_features).to(device)

    train_y = graph.y[graph.train_mask]
    counts = torch.bincount(train_y, minlength=2).float()
    weights = (counts.sum() / (2.0 * counts.clamp_min(1.0))).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
    best_state, best_val, best_epoch, wait = None, -np.inf, 0, 0

    for epoch in range(1, 501):
        model.train()
        optimizer.zero_grad()
        logits = model(graph.x, graph.edge_index)
        loss = F.cross_entropy(
            logits[graph.train_mask], graph.y[graph.train_mask], weight=weights
        )
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            logits = model(graph.x, graph.edge_index)
            prob = torch.softmax(logits, dim=1)[:, 1]
            val_true = graph.y[graph.val_mask].cpu().numpy()
            val_prob = prob[graph.val_mask].cpu().numpy()
            val_ap = average_precision_score(val_true, val_prob)

        if val_ap > best_val + 1e-6:
            best_val = val_ap
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if wait >= 40:
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        logits = model(graph.x, graph.edge_index)
        prob = torch.softmax(logits, dim=1)[:, 1]
        pred = torch.argmax(logits, dim=1)

    test_idx = torch.where(graph.test_mask)[0].cpu().numpy()
    result = pd.DataFrame({
        "Seed": seed,
        "Node_Index": test_idx,
        "Node_ID": nodes.iloc[test_idx]["Node_ID"].to_numpy(),
        "True_Label": graph.y[graph.test_mask].cpu().numpy(),
        "Predicted_Label": pred[graph.test_mask].cpu().numpy(),
        "AtRisk_Probability": prob[graph.test_mask].cpu().numpy(),
        "Best_Epoch": best_epoch
    })

    for name, col in ATTRIBUTES.items():
        result[name] = nodes.iloc[test_idx][col].fillna("Missing").astype(str).to_numpy()
    return result

all_predictions = []
for seed in SEEDS:
    print("Training seed", seed)
    all_predictions.append(train_seed(seed))

predictions_df = pd.concat(all_predictions, ignore_index=True)
print("Prediction rows:", len(predictions_df))

def safe_auc(y_true, prob, kind):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, prob) if kind == "roc" else average_precision_score(y_true, prob)

def metrics(y_true, prob, pred):
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    return {
        "N": len(y_true),
        "Positive_N": int((y_true == 1).sum()),
        "Negative_N": int((y_true == 0).sum()),
        "Selection_Rate": float((pred == 1).mean()),
        "AUC_ROC": safe_auc(y_true, prob, "roc"),
        "AUC_PR": safe_auc(y_true, prob, "pr"),
        "Precision_AtRisk": precision_score(y_true, pred, zero_division=0),
        "Recall_AtRisk": recall_score(y_true, pred, zero_division=0),
        "F1_AtRisk": f1_score(y_true, pred, zero_division=0),
        "False_Positive_Rate": fp/(fp+tn) if (fp+tn) else np.nan,
        "False_Negative_Rate": fn/(fn+tp) if (fn+tp) else np.nan,
        "Balanced_Accuracy": balanced_accuracy_score(y_true, pred),
        "Accuracy": accuracy_score(y_true, pred),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)
    }

group_rows = []
for seed, seed_df in predictions_df.groupby("Seed"):
    for attr in ATTRIBUTES:
        for group_name, g in seed_df.groupby(attr):
            group_rows.append({
                "Seed": seed, "Attribute": attr, "Group": group_name,
                **metrics(
                    g["True_Label"].to_numpy(),
                    g["AtRisk_Probability"].to_numpy(),
                    g["Predicted_Label"].to_numpy()
                )
            })

group_metrics_seed_df = pd.DataFrame(group_rows)

metric_cols = [
    "N","Positive_N","Negative_N","Selection_Rate","AUC_ROC","AUC_PR",
    "Precision_AtRisk","Recall_AtRisk","F1_AtRisk","False_Positive_Rate",
    "False_Negative_Rate","Balanced_Accuracy","Accuracy"
]

summary_rows = []
for (attr, group_name), g in group_metrics_seed_df.groupby(["Attribute","Group"]):
    row = {"Attribute": attr, "Group": group_name, "Seeds_Observed": g["Seed"].nunique()}
    for m in metric_cols:
        row[f"{m}_Mean"] = g[m].mean()
        row[f"{m}_SD"] = g[m].std(ddof=1)
    summary_rows.append(row)

group_summary_df = pd.DataFrame(summary_rows).sort_values(["Attribute","Group"])

gap_rows = []
for seed, seed_df in group_metrics_seed_df.groupby("Seed"):
    for attr, g in seed_df.groupby("Attribute"):
        eligible = g[g["N"] >= MIN_GROUP_TEST_N].copy()
        if len(eligible) < 2:
            continue
        dp = eligible["Selection_Rate"].max() - eligible["Selection_Rate"].min()
        eo = eligible["Recall_AtRisk"].max() - eligible["Recall_AtRisk"].min()
        fpr = eligible["False_Positive_Rate"].max() - eligible["False_Positive_Rate"].min()
        f1gap = eligible["F1_AtRisk"].max() - eligible["F1_AtRisk"].min()
        gap_rows.append({
            "Seed": seed,
            "Attribute": attr,
            "Eligible_Groups": len(eligible),
            "Demographic_Parity_Gap": dp,
            "Equal_Opportunity_Gap": eo,
            "False_Positive_Rate_Gap": fpr,
            "Equalized_Odds_Gap": max(eo, fpr),
            "F1_Gap": f1gap
        })

fairness_gaps_seed_df = pd.DataFrame(gap_rows)

gap_summary_rows = []
for attr, g in fairness_gaps_seed_df.groupby("Attribute"):
    row = {"Attribute": attr, "Seeds_Observed": g["Seed"].nunique()}
    for m in [
        "Demographic_Parity_Gap","Equal_Opportunity_Gap",
        "False_Positive_Rate_Gap","Equalized_Odds_Gap","F1_Gap"
    ]:
        row[f"{m}_Mean"] = g[m].mean()
        row[f"{m}_SD"] = g[m].std(ddof=1)
    gap_summary_rows.append(row)

fairness_gap_summary_df = pd.DataFrame(gap_summary_rows).sort_values(
    "Equalized_Odds_Gap_Mean", ascending=False
)

display(group_counts_df)
display(group_summary_df)
display(fairness_gap_summary_df)


In [ ]:

plt.figure(figsize=(9,5))
plot_df = fairness_gap_summary_df.reset_index(drop=True)
x = np.arange(len(plot_df))
w = 0.25
plt.bar(x-w, plot_df["Demographic_Parity_Gap_Mean"], w, label="Demographic parity")
plt.bar(x, plot_df["Equal_Opportunity_Gap_Mean"], w, label="Equal opportunity")
plt.bar(x+w, plot_df["False_Positive_Rate_Gap_Mean"], w, label="False-positive rate")
plt.xticks(x, plot_df["Attribute"], rotation=20)
plt.ylabel("Absolute gap")
plt.title("Baseline GraphSAGE fairness gaps")
plt.legend()
plt.tight_layout()
plt.show()


Larger gaps mean larger differences between groups. Small groups can produce unstable values, so always review sample sizes and standard deviations.

In [ ]:

group_counts_df.to_csv("fairness_group_counts.csv", index=False)
predictions_df.to_csv("fairness_baseline_predictions.csv", index=False)
group_metrics_seed_df.to_csv("fairness_group_metrics_by_seed.csv", index=False)
group_summary_df.to_csv("fairness_group_summary_mean_sd.csv", index=False)
fairness_gaps_seed_df.to_csv("fairness_gaps_by_seed.csv", index=False)
fairness_gap_summary_df.to_csv("fairness_gap_summary_mean_sd.csv", index=False)

files.download("fairness_group_counts.csv")
files.download("fairness_group_summary_mean_sd.csv")
files.download("fairness_gap_summary_mean_sd.csv")
files.download("fairness_gaps_by_seed.csv")
files.download("fairness_group_metrics_by_seed.csv")
files.download("fairness_baseline_predictions.csv")
